this script will:

- upload four geotiffs from google earth engine to google drive
- (then you manually download them locally)
- merges the four geotiffs into one
- convert from 32 bit to 8 bit and EPSG:4326 to EPSG:3857
- generate the tiles from the geotiff and put them in tiles/{x}/{y}/{z}


In [ ]:
import ee

ee.Initialize(project="gsapp-map")

img = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterDate("2024-01-01", "2024-12-31")
    .mosaic()
    .select(["A01", "A02", "A03"])
)

regions = [
    ee.Geometry.Rectangle([-180, -85, -90, 85]),
    ee.Geometry.Rectangle([-90, -85, 0, 85]),
    ee.Geometry.Rectangle([0, -85, 90, 85]),
    ee.Geometry.Rectangle([90, -85, 180, 85]),
]

for i, reg in enumerate(regions):
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f"alphaearth_2024_rgb_part{i}",
        folder="AlphaEarthTiles3",
        fileNamePrefix=f"alphaearth_2024_rgb_part{i}",
        region=reg,
        scale=10000,
        crs="EPSG:4326",
        maxPixels=1e13,
    )
    task.start()
    print(f"Started export for region {i}")

In [ ]:
import glob
import os
import subprocess

# --- Step 0: Set file paths ---
input_files = sorted(glob.glob("alphaearth_2024_rgb_part*.tif"))
merged_file = "alphaearth_2024_merged.tif"
merged_8bit_file = "alphaearth_2024_merged_8bit.tif"
warped_file = "alphaearth_2024_merged_8bit_warped.tif"
tiles_dir = "../tiles"

# --- Step 1: Merge all parts into one ---
print("Merging TIFF parts...")
merge_cmd = ["gdal_merge.py", "-o", merged_file] + input_files
subprocess.run(merge_cmd, check=True)
print(f"Merged file created: {merged_file}")

# --- Step 1b: sanity check ---
print("Checking merged file info...")
subprocess.run(["gdalinfo", merged_file])

# --- Step 2: Convert to 8-bit RGB ---
print("Converting to 8-bit RGB...")
translate_cmd = [
    "gdal_translate",
    "-ot",
    "Byte",
    "-scale",
    "-0.3",
    "0.3",
    "0",
    "255",
    "-co",
    "PHOTOMETRIC=RGB",
    merged_file,
    merged_8bit_file,
]
subprocess.run(translate_cmd, check=True)
print(f"8-bit RGB file created: {merged_8bit_file}")

# --- Step 3: Warp to Web Mercator for tile generation ---
print("Warping to Web Mercator (EPSG:3857)...")
warp_cmd = [
    "gdalwarp",
    "-t_srs",
    "EPSG:3857",
    "-te",
    "-20037508.3427892",
    "-20037508.3427892",
    "20037508.3427892",
    "20037508.3427892",
    "-tr",
    "5000",
    "5000",  # adjust resolution if desired
    "-r",
    "bilinear",
    merged_8bit_file,
    warped_file,
]
subprocess.run(warp_cmd, check=True)
print(f"Warped file created: {warped_file}")

# --- Step 3b: sanity check ---
print("Checking warped file info...")
subprocess.run(["gdalinfo", warped_file])

# --- Step 4: Generate XYZ tiles for MapLibre ---
print("Generating XYZ tiles for MapLibre...")
os.makedirs(tiles_dir, exist_ok=True)
tiles_cmd = ["gdal2tiles.py", "--xyz", "-z", "0-5", warped_file, tiles_dir]
subprocess.run(tiles_cmd, check=True)
print(f"Tiles generated in folder: {tiles_dir}")

🧩 Merging TIFF parts...
0...10...20...30...40...50...60...70...80...90...100 - done.
✅ Merged file created: alphaearth_2024_merged.tif
🔍 Checking merged file info...
Driver: GTiff/GeoTIFF
Files: alphaearth_2024_merged.tif
Size is 4008, 1926
Coordinate System is:
GEOGCRS["WGS 84",
    ENSEMBLE["World Geodetic System 1984 ensemble",
        MEMBER["World Geodetic System 1984 (Transit)"],
        MEMBER["World Geodetic System 1984 (G730)"],
        MEMBER["World Geodetic System 1984 (G873)"],
        MEMBER["World Geodetic System 1984 (G1150)"],
        MEMBER["World Geodetic System 1984 (G1674)"],
        MEMBER["World Geodetic System 1984 (G1762)"],
        MEMBER["World Geodetic System 1984 (G2139)"],
        MEMBER["World Geodetic System 1984 (G2296)"],
        ELLIPSOID["WGS 84",6378137,298.257223563,
            LENGTHUNIT["metre",1]],
        ENSEMBLEACCURACY[2.0]],
    PRIMEM["Greenwich",0,
        ANGLEUNIT["degree",0.0174532925199433]],
    CS[ellipsoidal,2],
        AXIS["geode

Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done in 00:01:35.


Generating Overview Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:25.
✅ Tiles generated in folder: tiles

🚀 All steps complete! You can now serve the 'tiles' folder with MapLibre.
